## Sieć neuronowa do rozpoznawania instrumentów

### Biblioteki

In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score

### Pobranie danych wejściowych z mel-spektogramu

In [2]:
main = Path("../..").resolve()
spectrograms = main / Path("spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s")
labels = spectrograms / "etykiety.csv"

data = pd.read_csv(labels)

In [3]:
print(data.shape)
print(data.columns.tolist())

(11011, 9)
['dataset', 'audio_file', 'Acoustic Guitar (steel)', 'Alto Sax', 'Bright Acoustic Piano', 'Drums', 'Electric Bass (finger)', 'Flute', 'String Ensemble 1']


In [4]:
print(list(spectrograms.iterdir()))

[WindowsPath('C:/Users/Wiktor/Documents/GitHub/instrument-recognition-project/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/etykiety.csv'), WindowsPath('C:/Users/Wiktor/Documents/GitHub/instrument-recognition-project/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/test'), WindowsPath('C:/Users/Wiktor/Documents/GitHub/instrument-recognition-project/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/train'), WindowsPath('C:/Users/Wiktor/Documents/GitHub/instrument-recognition-project/spectrograms/mels80_frame1024_step512_windowhann_sr44100_4s/validation')]


In [5]:
instrumenty = [
    "Acoustic Guitar (steel)",
    "Alto Sax",
    "Bright Acoustic Piano",
    "Drums",
    "Electric Bass (finger)",
    "Flute",
    "String Ensemble 1"
]

In [6]:
for zbior in ["train", "validation", "test"]:
    folder = spectrograms / zbior
    pliki = list(folder.iterdir())

    print(f"\n{zbior}: {len(pliki)} elementów")
    print("Przykładowe pliki:", [p.name for p in pliki[:5]])


train: 7709 elementów
Przykładowe pliki: ['Track00001_204_5260.npy', 'Track00001_232_1903.npy', 'Track00003_32_1841.npy', 'Track00005_108_1081.npy', 'Track00005_108_3995.npy']

validation: 1646 elementów
Przykładowe pliki: ['Track00006_108_8566.npy', 'Track00006_228_1616.npy', 'Track00006_236_1661.npy', 'Track00007_12_5311.npy', 'Track00007_32_5852.npy']

test: 1656 elementów
Przykładowe pliki: ['Track00003_24_948.npy', 'Track00004_56_327.npy', 'Track00005_144_4050.npy', 'Track00005_200_583.npy', 'Track00005_24_3997.npy']


In [7]:
# Sprawdzamy, czy nazwy plików NPY odpowiadają nazwom z CSV

for zbior in ["train", "validation", "test"]:
    pliki_npy = {
        p.stem
        for p in (spectrograms / zbior).glob("*.npy")
    }

    pliki_csv = set(
        data.loc[data["dataset"] == zbior, "audio_file"]
        .str.replace(".flac", "", regex=False)
    )

    print(f"\n{zbior}")
    print("Plików NPY:", len(pliki_npy))
    print("Rekordów CSV:", len(pliki_csv))
    print("Wspólnych nazw:", len(pliki_npy & pliki_csv))


train
Plików NPY: 7709
Rekordów CSV: 7709
Wspólnych nazw: 7709

validation
Plików NPY: 1646
Rekordów CSV: 1646
Wspólnych nazw: 1646

test
Plików NPY: 1656
Rekordów CSV: 1656
Wspólnych nazw: 1656


In [8]:
class InstrumentDataset(Dataset):
    def __init__(self, data, root, zbior):
        self.data = (
            data[data["dataset"] == zbior]
            .reset_index(drop=True)
        )
        self.root = root
        self.zbior = zbior

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Ścieżka do pliku spektrogramu
        nazwa_pliku = Path(row["audio_file"]).with_suffix(".npy").name
        sciezka = self.root / self.zbior / nazwa_pliku

        # Wczytanie mel-spektrogramu
        spec = np.load(sciezka)
        spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0)

        # Etykiety instrumentów
        label = row[instrumenty].to_numpy(dtype=np.float32)
        label = torch.tensor(label, dtype=torch.float32)

        return spec, label

In [9]:
train_dataset = InstrumentDataset(data, spectrograms, "train")
val_dataset = InstrumentDataset(data, spectrograms, "validation")
test_dataset = InstrumentDataset(data, spectrograms, "test")

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False)

### Wybór urządzenia do obliczeń dla modelu

In [10]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    return device

### Architektura sieci

In [11]:
# jest to wstępna, jak będą dane z mel-spektogramu to się rozbuduje sieć, aby jak najlepiej zoptymalizować ją

class Instrument_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,32,3,1)
        self.conv2 = nn.Conv2d(32,64,3,1)
        self.conv3 = nn.Conv2d(64,128,3,1)
        self.dropout2 = nn.Dropout2d(0.4)
        self.dropout3 = nn.Dropout2d(0.6)

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc3 = nn.Linear(128, 7)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)

        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout2(x)

        x = self.conv3(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout3(x)

        x = self.global_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc3(x)

        return x


In [12]:
device = get_device()

model = Instrument_CNN().to(device)

criterion = nn.MultiLabelSoftMarginLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Urządzenie:", device)

Urządzenie: cuda


In [13]:
print("Liczba próbek:", len(train_dataset))
print("Pierwszy rekord:")
print(train_dataset.data.iloc[0])

spec, label = train_dataset[0]

print("Kształt spektrogramu:", spec.shape)
print("Kształt etykiet:", label.shape)

Liczba próbek: 7709
Pierwszy rekord:
dataset                                      train
audio_file                 Track01881_24_1982.flac
Acoustic Guitar (steel)                          0
Alto Sax                                         0
Bright Acoustic Piano                            1
Drums                                            0
Electric Bass (finger)                           1
Flute                                            0
String Ensemble 1                                0
Name: 0, dtype: object
Kształt spektrogramu: torch.Size([1, 80, 346])
Kształt etykiet: torch.Size([7])


In [14]:
X, y = next(iter(train_loader))

print("Kształt spektrogramów:", X.shape)
print("Kształt etykiet:", y.shape)
print("Przykładowe etykiety:", y[0])

Kształt spektrogramów: torch.Size([4, 1, 80, 346])
Kształt etykiet: torch.Size([4, 7])
Przykładowe etykiety: tensor([1., 0., 1., 1., 1., 0., 1.])


In [20]:
epochs = 100

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    for X, y in train_loader:
        X = X.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(X)
        loss = criterion(outputs, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(device)
            y = y.to(device)

            outputs = model(X)
            loss = criterion(outputs, y)

            val_loss += loss.item() * X.size(0)

    val_loss /= len(val_loader.dataset)

    print(
        f"Epoka {epoch + 1}/{epochs} | "
        f"Train loss: {train_loss:.4f} | "
        f"Val loss: {val_loss:.4f}"
    )


Epoka 1/100 | Train loss: 0.4466 | Val loss: 0.4137
Epoka 2/100 | Train loss: 0.4394 | Val loss: 0.4073
Epoka 3/100 | Train loss: 0.4392 | Val loss: 0.4057
Epoka 4/100 | Train loss: 0.4375 | Val loss: 0.4047
Epoka 5/100 | Train loss: 0.4363 | Val loss: 0.4138
Epoka 6/100 | Train loss: 0.4296 | Val loss: 0.3889
Epoka 7/100 | Train loss: 0.4288 | Val loss: 0.4066
Epoka 8/100 | Train loss: 0.4258 | Val loss: 0.3986


KeyboardInterrupt: 

In [ ]:
model.eval()

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        prawdopodobienstwa = torch.sigmoid(logits)
        predykcje = (prawdopodobienstwa >= 0.5).float()

        print("Prawdopodobieństwa:")
        print(prawdopodobienstwa.cpu())

        print("\nPrzewidywane etykiety:")
        print(predykcje.cpu())

        print("\nPrawdziwe etykiety:")
        print(y.cpu())

        break


Prawdopodobieństwa:
tensor([[0.7024, 0.4467, 0.6587, 0.9998, 0.9870, 0.4832, 0.6116],
        [0.5380, 0.3429, 0.8932, 1.0000, 0.4897, 0.4033, 0.5310],
        [0.0540, 0.4103, 0.1511, 1.0000, 0.0058, 0.5507, 0.2896],
        [0.3330, 0.4268, 0.4770, 1.0000, 0.0991, 0.4895, 0.7115]])

Przewidywane etykiety:
tensor([[1., 0., 1., 1., 1., 0., 1.],
        [1., 0., 1., 1., 0., 0., 1.],
        [0., 0., 0., 1., 0., 1., 0.],
        [0., 0., 0., 1., 0., 0., 1.]])

Prawdziwe etykiety:
tensor([[1., 0., 1., 1., 1., 1., 1.],
        [0., 1., 1., 1., 0., 0., 0.],
        [0., 1., 0., 1., 0., 1., 0.],
        [0., 1., 0., 1., 0., 0., 1.]])


In [ ]:
# Zapis wag modelu
torch.save(model.state_dict(), "model_weights.pth")

# Ewaluacja na zbiorze testowym
model.eval()

wszystkie_predykcje = []
wszystkie_etykiety = []

with torch.no_grad():
    for X, y in test_loader:
        X = X.to(device)
        y = y.to(device)

        logits = model(X)
        prawdopodobienstwa = torch.sigmoid(logits)
        predykcje = (prawdopodobienstwa >= 0.5).float()

        wszystkie_predykcje.extend(predykcje.cpu().numpy().flatten())
        wszystkie_etykiety.extend(y.cpu().numpy().flatten())

# Obliczenie F1-score
f1 = f1_score(
    wszystkie_etykiety,
    wszystkie_predykcje,
    zero_division=0
)

print(f"F1-score na zbiorze testowym: {f1:.4f}")

Wagi modelu zapisano do pliku model_weights.pth
F1-score na zbiorze testowym: 0.7931
